In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch
import torchvision
from sklearn.metrics import accuracy_score

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [3]:
class ImageDataset(Dataset):
    def __init__(self, image_dir, csv_path, transform=None, label_col=True):
        """
        Args:
            image_dir (str): Directory with training images.
            csv_path (str): Path to CSV with image filenames and labels.
            transform (callable, optional): Optional transform to apply to each image.
        """
        self.image_dir = image_dir
        self.labels_df = pd.read_csv(csv_path)
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        # Get filename and label
        img_name = self.labels_df.iloc[idx, 0]
        if self.label_col:
            label = int(self.labels_df.iloc[idx, 1])
        else:
            label = 0

        # Build full path and open image
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        # Apply any transforms
        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
image_dir = "image-classification-real-or-ai-generated-photo/train/train"
csv_path = "image-classification-real-or-ai-generated-photo/train.csv"

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])
])

In [6]:
dataset = ImageDataset(image_dir=image_dir, csv_path=csv_path, transform=transform)

In [7]:
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2])

In [8]:
mini_batch_size = 32

In [9]:
train_dl = DataLoader(train_dataset, batch_size=mini_batch_size, shuffle=True, drop_last=False)
val_dl = DataLoader(val_dataset, batch_size=mini_batch_size * 2, drop_last=False)

In [10]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [11]:
def fit(model, optimizer, early_stopping, train_dl, valid_dl):
    loss_func = torch.nn.CrossEntropyLoss()

    # loop over epochs
    for epoch in range(100):
        model.train()

        # loop over mini-batches
        for X_mb, y_mb in train_dl:
            X_mb = X_mb.to(device)
            y_mb = y_mb.to(device)
            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            train_loss = sum(loss_func(model(X_mb.to(device)), y_mb.to(device)) for X_mb, y_mb in train_dl)
            valid_loss = sum(loss_func(model(X_mb.to(device)), y_mb.to(device)) for X_mb, y_mb in valid_dl)
        print('epoch {}, training loss {}'.format(epoch + 1, train_loss / len(train_dl)))
        print('epoch {}, validation loss {}'.format(epoch + 1, valid_loss / len(valid_dl)))

        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

In [12]:
model = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")

# Replace final classification head
num_features = model.classifier[2].in_features
model.classifier[2] = torch.nn.Linear(num_features, 2)  # binary output

In [13]:
# model = torchvision.models.swin_t(weights="IMAGENET1K_V1")

# # Replace final classification head
# num_features = model.head.in_features
# model.head = torch.nn.Linear(num_features, 2)  # binary output

In [14]:
model

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [15]:
model = model.to(device)

In [16]:
optimizer = torch.optim.Adam(model.parameters())
early_stopping = EarlyStopping(patience=10, delta=0.01)

In [17]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

27821666

In [18]:
fit(model, optimizer, early_stopping, train_dl, val_dl)

epoch 1, training loss 0.6153993010520935
epoch 1, validation loss 0.6057552695274353
epoch 2, training loss 0.4727865755558014
epoch 2, validation loss 0.4514625072479248
epoch 3, training loss 0.48262909054756165
epoch 3, validation loss 0.44884800910949707
epoch 4, training loss 0.29376640915870667
epoch 4, validation loss 0.3030959963798523
epoch 5, training loss 0.1326107531785965
epoch 5, validation loss 0.21463477611541748
epoch 6, training loss 0.4205108880996704
epoch 6, validation loss 0.5126538276672363
epoch 7, training loss 0.05954610928893089
epoch 7, validation loss 0.3664102554321289
epoch 8, training loss 0.055251363664865494
epoch 8, validation loss 0.30394020676612854
epoch 9, training loss 0.051558688282966614
epoch 9, validation loss 0.1887407749891281
epoch 10, training loss 0.15743215382099152
epoch 10, validation loss 0.4759604036808014
epoch 11, training loss 0.25266483426094055
epoch 11, validation loss 0.5688801407814026
epoch 12, training loss 0.011023577302

In [19]:
early_stopping.load_best_model(model)

In [20]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():  # disable gradient tracking
    for X_batch, y_batch in train_dl:
        outputs = model(X_batch.to(device)).cpu()
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.tolist())
        true_labels.extend(y_batch.tolist())

In [21]:
accuracy_score(predictions, true_labels)

0.9986772486772487

In [22]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():  # disable gradient tracking
    for X_batch, y_batch in val_dl:
        outputs = model(X_batch.to(device)).cpu()
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.tolist())
        true_labels.extend(y_batch.tolist())

In [23]:
accuracy_score(predictions, true_labels)

0.9206349206349206

In [24]:
test_dataset = ImageDataset(image_dir="image-classification-real-or-ai-generated-photo/test/test", csv_path="image-classification-real-or-ai-generated-photo/test.csv", transform=transform, label_col=False)
test_dl = DataLoader(test_dataset, batch_size=mini_batch_size * 2, drop_last=False)

In [25]:
model.eval()
predictions = []

with torch.no_grad():  # disable gradient tracking
    for X_batch, y_batch in test_dl:
        outputs = model(X_batch.to(device)).cpu()
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.tolist())

In [26]:
test_csv = pd.read_csv("image-classification-real-or-ai-generated-photo/test.csv")
test_csv["Label"] = predictions

In [27]:
test_csv

,Image,Label
0,946.jpg,0
1,947.jpg,1
2,948.jpg,0
3,949.jpg,1
4,950.jpg,0
...,...,...
397,1343.jpg,1
398,1344.jpg,1
399,1345.jpg,1
400,1346.jpg,1


In [28]:
test_csv.Label.mean()

np.float64(0.4626865671641791)

In [29]:
test_csv.to_csv("submission.csv", index=False)